In [2]:
%pip install -U anthropic
%pip install -U brotli

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# ReAct Pattern

ReAct stands for Reason Action and the pattern can be defined by a reason, action, observation loop. Within this pattern the agent is instructed to first reason about how to solve the problem then it defines an action which will either be answering the question or calling a tool. If the agent chooses to call a tool the results will then be added as an observation. The easiest way I have found to do this is by instructing the agent to wrap all text within XML tags and extract everything within those tags. An example output would be <thought> The agent thinking about what to do</thought> <tool_call>Agent calling a tool</tool_call> then the output of the tool would we added to context like <observation>tool result</observation> and finally when the agent is finished it returns <response> The answer</response>. This pattern is perfect for problems which can be broken down into smaller simpler subproblems. It's also dynamic compared to chaining which requires the exact sequence to be defined before a query is ever processed.

In [3]:
from anthropic import Anthropic
import os
from IPython.display import display_markdown

client = Anthropic(
    api_key= os.environ.get("ANTHROPIC_API_KEY")
)

MODEL = 'claude-haiku-5-5'

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.patterns import ReactAgent, tool

## Defining tools the Agent has access to 

In [5]:
import math

@tool
def sum_two_elements(a: int, b: int) -> int:
    """
    Computes the sum of two integers.

    Args:
        a (int): The first integer to be summed.
        b (int): The second integer to be summed.

    Returns:
        int: The sum of `a` and `b`.
    """
    return a + b


@tool
def multiply_two_elements(a: int, b: int) -> int:
    """
    Multiplies two integers.

    Args:
        a (int): The first integer to multiply.
        b (int): The second integer to multiply.

    Returns:
        int: The product of `a` and `b`.
    """
    return a * b

@tool
def compute_log(x: int) -> float | str:
    """
    Computes the logarithm of an integer `x` with an optional base.

    Args:
        x (int): The integer value for which the logarithm is computed. Must be greater than 0.

    Returns:
        float: The logarithm of `x` to the specified `base`.
    """
    if x <= 0:
        return "Logarithm is undefined for values less than or equal to 0."
    
    return math.log(x)


available_tools = {
    "sum_two_elements": sum_two_elements,
    "multiply_two_elements": multiply_two_elements,
    "compute_log": compute_log
}

tool_list = [sum_two_elements, multiply_two_elements, compute_log]

In [6]:
agent = ReactAgent(tools=tool_list, model_name=MODEL, client=client)


In [7]:
result = agent.run(user_prompt="I want to calculate the sum of 1234 and 5678 and multiply the result by 5. Then, I want to take the logarithm of this result")

## Step Analysis
Looking at the steps the Agent took you can see how it was able to break the problem down into smaller steps and leverage the tools provided in order to answer the question. This is a simpler question but from the results you can see why the ReAct pattern can be extremely powerful when used to breakdown more complex problems.

In [10]:
for step in result.steps:
    
    print("Thought:")
    display_markdown(step.thought, raw=True)
    print("Tool Call:")
    display_markdown(step.tool_call, raw=True)

Thought:


First, I need to compute the sum of 1234 and 5678.

Tool Call:


{"name":"sum_two_elements","arguments":{"a":1234,"b":5678},"id":0}

Thought:


...

Tool Call:


{"name":"compute_log","arguments":{"x":34560},"id":2}

Thought:


Tool Call:


The sum of 1234 and 5678 is 6912. Multiplying that by 5 gives 34560. The logarithm of 34560 is approximately 10.4505.

In [11]:
display_markdown(result.resp, raw=True)

The sum of 1234 and 5678 is 6912. Multiplying that by 5 gives 34560. The logarithm of 34560 is approximately 10.4505.

## No ReAct Results

You can see that the model is able to achieve the correct answer even without the ReAct loop and tools. This shows that the model was extremely well trained and is very powerful. You can see from the output that the model followed the same steps used within the ReAct loop as it did within it's prompt.

In [12]:
resp = client.messages.create(
    max_tokens=1000,
    model = MODEL,
    messages=[{'role': 'user', 'content': 'I want to calculate the sum of 1234 and 5678 and multiply the result by 5. Then, I want to take the logarithm of this result'}]
)

text = "".join([b.text for b in resp.content if b.type == 'text'])

display_markdown(text, raw=True)

**Step 1: Sum**
1234 + 5678 = 6912

**Step 2: Multiply by 5**
6912 × 5 = 34560

**Step 3: Logarithm of 34560**

- Common logarithm (base 10): log₁₀(34560) ≈ **4.5386**
- Natural logarithm (base e): ln(34560) ≈ **10.4505**

If you had a specific base in mind, let me know and I can compute that instead.